In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from collections import Counter
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.multioutput import ClassifierChain
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = "data.csv"   # keep the CSV in the same folder as the notebook
df = pd.read_csv(file_path)

# All feature columns (exclude label)
feature_cols = [col for col in df.columns if col != 'class_label']

# Create signature by combining all features
df['row_signature'] = df[feature_cols].astype(str).agg('_'.join, axis=1)


# One-hot encode labels
one_hot = pd.get_dummies(df['class_label'])

# Combine with signature
df_multi = pd.concat([df['row_signature'], one_hot], axis=1)

# Merge duplicates (same pair → combine labels)
df_multi = df_multi.groupby('row_signature').max().reset_index()

# Take first occurrence of each pair
X = df.groupby('row_signature')[feature_cols].first().reset_index(drop=True)

# Multilabel targets
Y = df_multi.drop(columns=['row_signature'])

print("X shape:", X.shape)
print("Y shape:", Y.shape)

# Check multilabel nature
print("\nLabels per sample:")
print(Y.sum(axis=1).value_counts())

# Combine X and Y into one DataFrame
#df_final = pd.concat([X.reset_index(drop=True), Y.reset_index(drop=True)], axis=1)

# Save to CSV
#df_final.to_csv("multilabel_dataset.csv", index=False)

#print("Saved as multilabel_dataset.csv")

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, test_idx in mskf.split(X, Y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    break

# -------------------------------
# Random Forest Model
# -------------------------------
base_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',   # 🔥 handles imbalance
    n_jobs=-1,
    random_state=42
)

model = ClassifierChain(
    base_estimator=base_model,
    order='random',
    random_state=42
)

model.fit(X_train, Y_train)

# -------------------------------
# Prediction (same as before)
# -------------------------------
probs_list = []

for p in model.predict_proba(X_test):
    if len(p.shape) == 2:
        probs_list.append(p[:, 1])
    else:
        probs_list.append(p)

Y_pred_prob = np.column_stack(probs_list)

# Fix orientation if needed
if Y_pred_prob.shape[0] == Y.shape[1]:
    Y_pred_prob = Y_pred_prob.T

# -------------------------------
# Thresholding
# -------------------------------
thresholds = Y_train.mean(axis=0).values
Y_pred = (Y_pred_prob >= thresholds).astype(int)

# -------------------------------
# Evaluation
# -------------------------------
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Subset Accuracy:", accuracy_score(Y_test, Y_pred))

print("\n--- Micro ---")
print("Precision:", precision_score(Y_test, Y_pred, average='micro', zero_division=0))
print("Recall:", recall_score(Y_test, Y_pred, average='micro', zero_division=0))
print("F1:", f1_score(Y_test, Y_pred, average='micro', zero_division=0))

print("\n--- Macro ---")
print("Precision:", precision_score(Y_test, Y_pred, average='macro', zero_division=0))
print("Recall:", recall_score(Y_test, Y_pred, average='macro', zero_division=0))
print("F1:", f1_score(Y_test, Y_pred, average='macro', zero_division=0))

X shape: (46238, 126)
Y shape: (46238, 7)

Labels per sample:
1    32028
2     8834
3     4223
4      742
5      221
6      190
Name: count, dtype: int64
Subset Accuracy: 0.6170005414185166

--- Micro ---
Precision: 0.726933669956392
Recall: 0.9372641858400532
F1: 0.8188075618031991

--- Macro ---
Precision: 0.701798850026414
Recall: 0.9380462279673366
F1: 0.7973803432332153


In [ ]:
import pandas as pd
import numpy as np

# Convert X to DataFrame (important for debugging)
X_df = pd.DataFrame(X_train)

# Try numeric conversion
X_numeric = X_df.apply(pd.to_numeric, errors='coerce')

# Find bad cells (where conversion failed)
bad_mask = X_numeric.isna() & X_df.notna()

# Rows with any bad values
bad_rows = bad_mask.any(axis=1)

print(f"Number of problematic rows: {bad_rows.sum()}")

# Show problematic rows
print("\nProblematic rows:")
print(X_df[bad_rows].head())

# Show exact bad values + column index
for row_idx in np.where(bad_rows)[0][:5]:
    for col_idx in np.where(bad_mask.iloc[row_idx])[0]:
        print(f"Row {row_idx}, Column {col_idx}: {X_df.iloc[row_idx, col_idx]}")